## Libraries

In [ ]:
from utils.visualization import *
from utils.dataset import *
from utils.miscellaneous import *
from utils.scaling import *
from models.gnn import *
from models.models import *
from training.train import *
from training.loss import *
from database.graph_creation import *
from database.dhydro_utils import *

### Plot details

In [ ]:
import matplotlib as mpl

mpl.rcParams['grid.color'] = 'k'
mpl.rcParams['grid.linestyle'] = ':'
mpl.rcParams['grid.linewidth'] = 0.5

mpl.rcParams['figure.figsize'] = [7, 5]
mpl.rcParams['figure.dpi'] = 100
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['savefig.bbox'] = 'tight'

mpl.rcParams['font.size'] = 18
mpl.rcParams['legend.fontsize'] = 'small'
mpl.rcParams['figure.titlesize'] = 'medium'

mpl.rcParams['font.family'] = 'serif'

## Dataset creation

In [ ]:
cfg_file = "config_test.yaml"
config = read_config(cfg_file)

save_folder = config.get('save_folder', 'results')
os.makedirs(save_folder, exist_ok=True)

case_study = 'dk41'
dataset_folder = f"database/raw_datasets_{case_study}"

with_polygon_mesh = config['dataset_parameters'].get('dataset_folder').split('/')[-1] == 'polygon_meshes'

### Setup case study and mesh

In [ ]:
breach_id = config.get('breach_location', 'all')
return_period = config.get('return_period', None)

save_folder = os.path.join(config.get('save_folder', 'results'), breach_id)
os.makedirs(save_folder, exist_ok=True)

time_stop = 64 # num time steps (64 * 8h)
type_BC = 2 # discharge BC

In [ ]:
# plot dike ring area
dijkpalen_file = os.path.join(dataset_folder, "dijkpalen.gpkg")
dijkpalen = gpd.read_file(dijkpalen_file)
dijkpalen.sort_values(by='CODE', inplace=True)
dijkpalen = dijkpalen[dijkpalen.WS_DIJKRIN == 'DR41']
dijkpalen_coords = np.array([[geom.x, geom.y] for geom in dijkpalen.geometry])

if breach_id != 'all':
    if isinstance(breach_id, list):
        breach_index = np.where(dijkpalen.CODE.isin([b + '.' for b in breach_id]))[0]
    else:
        breach_index = np.where(dijkpalen.CODE == (breach_id + '.'))[0]
    BC_loc = np.array([(geom.x, geom.y) for geom in dijkpalen.iloc[breach_index].geometry])
else:
    selected_dijkpalen_HD = dijkpalen[dijkpalen.CODE.str.startswith(('HD'))][::9]
    selected_dijkpalen_ND = dijkpalen[dijkpalen.CODE.str.startswith(('ND'))][::8][::-1]

    selected_dijkpalen = pd.concat([selected_dijkpalen_HD, selected_dijkpalen_ND])
    BC_loc = np.array([(geom.x, geom.y) for geom in selected_dijkpalen.geometry])

In [ ]:
plt.figure(figsize=(10, 5))

polygons_file = os.path.join(dataset_folder, "Geometry", f"dense_simplified_polygons.gpkg")
boundary_polygon = gpd.read_file(polygons_file).union_all()
boundary_coords = np.array(boundary_polygon.boundary.coords)

plt.plot(*boundary_coords.T)
plt.gca().set_aspect('equal')
correct_plt_units(plt.gca(), boundary_coords)

# plot points
print("Number of points: ", len(BC_loc))
plt.scatter(BC_loc[:,0], BC_loc[:,1], s=50, c='r', marker='x', zorder=2);
# for i, txt in enumerate(range(len(BC_loc))):
#     plt.annotate(txt, (BC_loc[i,0], BC_loc[i,1]))

### Create meshes

This part can/should largely be optimized. So far, I'm creating a ghost cell at the finest mesh and re-connecting all of the scales together: this takes a lot of time due to the number of cells, but in principle the only connections that change are the ones at the ghost cell boundary, but it works, so I didn't further optimize it, sorry :D

In [ ]:
new_mesh_file = os.path.join(save_folder, "base_meshes.pkl")
mesh_file = os.path.join(dataset_folder, "base_meshes.pkl")

if not os.path.exists(new_mesh_file):
    base_datas = []
    for bc_loc in tqdm(BC_loc, total=len(BC_loc)):
        j = 1
        while True:
            with open(mesh_file, 'rb') as f:
                meshes = pickle.load(f)
                if with_polygon_mesh:
                    meshes.pop(2)
                    meshes.pop(2)
                    meshes.pop(2)
                else:
                    meshes.pop(0)
                    meshes.pop(0)
            base_data = add_BC_to_data(copy(meshes), np.array([bc_loc]), np.zeros(time_stop), type_BC=2)
            if base_data:
                _, closest_idx = cKDTree(dijkpalen_coords).query(bc_loc)
                base_data.CODE = dijkpalen.iloc[closest_idx].CODE
                base_datas.append(base_data)
                break
            else:
                _, closest_idx = cKDTree(dijkpalen_coords).query(bc_loc, k=j+1)
                closest_dijkpalen = dijkpalen.iloc[closest_idx[j:]]
                bc_loc = np.array([(geom.x, geom.y) for geom in closest_dijkpalen.geometry]).squeeze()
                j += 1

    mesh_counts = np.unique_counts([data.DEM.shape for data in base_datas])
    best_cell_num = mesh_counts[0][mesh_counts[1].argmax()]

    base_datas = [data for data in base_datas if data.DEM.shape[0] == best_cell_num]
        
    with open(new_mesh_file, 'wb') as f:
        pickle.dump(base_datas, f)
else:
    with open(new_mesh_file, 'rb') as f:
        base_datas = pickle.load(f)
    
with open(mesh_file, 'rb') as f:
    meshes = pickle.load(f)
    if with_polygon_mesh:
        meshes.pop(2)
        meshes.pop(2)
        meshes.pop(2)
    else:
        meshes.pop(0)
        meshes.pop(0)

num_breaches = len(base_datas)
print(f'Number of breach locations: {num_breaches}')

In [ ]:
num_scenarios_per_location = config.get('num_scenarios_per_location', 20)
num_scenarios = num_breaches * num_scenarios_per_location

# generate BCs
peak_value=(6.2, 0.4) # loc, scale of the lognormal distribution
peak_value=(100, 1500) # uniform distribution (change below)
min_discharge=0
param1=(1, 0.4)
param2=(0., 0.2)
param3=(5,15)
param4=0.6
x_window=(0.2, 1.6)

all_hydrographs = []
for i in range(num_scenarios):
   np.random.seed(i)

   hydrograph_time_series = generate_realistic_hydrograph(time_stop-1, 1, 
                                                         np.random.uniform(*peak_value),
                                                         min_discharge, 
                                                         np.random.lognormal(*param1), 
                                                         np.random.uniform(*param2), 
                                                         np.random.uniform(*param3), 
                                                         param4,
                                                         x_window)
   all_hydrographs.append(hydrograph_time_series[1])
   plt.plot(hydrograph_time_series[1])

all_hydrographs = np.array(all_hydrographs)
all_hydrographs[all_hydrographs < 1e-10] = 0

In [ ]:
torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision('high')  # allows TF32 accumulation in matmul

L.seed_everything(config['models']['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset_parameters = config['dataset_parameters']
selected_node_features = config['selected_node_features']
selected_edge_features = config['selected_edge_features']

with_polygon_mesh = dataset_parameters.get('dataset_folder').split('/')[-1] == 'polygon_meshes'
temporal_dataset_parameters = config['temporal_dataset_parameters']

scalers = copy(config['scalers'])
scalers = get_scalers(base_datas, scalers)

# Model

## Creation

In [ ]:
temporal_res = dataset_parameters['temporal_res']
previous_t = temporal_dataset_parameters['previous_t']
future_t = temporal_dataset_parameters['future_t']
time_start = temporal_dataset_parameters['time_start']
max_rollout_steps = temporal_dataset_parameters['rollout_steps']
test_dataset_name = dataset_parameters['test_dataset_name']

num_node_features = NUM_WATER_VARS*previous_t + sum(selected_node_features.values())
num_edge_features = sum(selected_edge_features.values())

print('Number of node features:\t', num_node_features)
print('Number of rollout steps:\t', max_rollout_steps)

In [ ]:
model_parameters = copy(config['models'])
model_type = model_parameters.pop('model_type')

if model_type == 'MSGNN':
    num_scales = len(meshes)
    model_parameters['num_scales'] = num_scales
    
model = get_model(model_type)(
    num_node_features=num_node_features,
    num_edge_features=num_edge_features,
    previous_t=previous_t,
    future_t=future_t,
    **model_parameters)

In [ ]:
trainer_options = copy(config['trainer_options'])
lr_info = config['lr_info']

# info for testing dataset
temporal_test_dataset_parameters = get_temporal_test_dataset_parameters(config, temporal_dataset_parameters)

print("Total Number of paramters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# Load trained model
plmodule_kwargs = {'model': model, 
                   'lr_info': lr_info, 
                   'trainer_options': trainer_options, 
                   'temporal_test_dataset_parameters': temporal_test_dataset_parameters}

if 'saved_model' in config:
    plmodel = LightningTrainer.load_from_checkpoint(config['saved_model'], **plmodule_kwargs)
    model = plmodel.model
else:
    plmodel = LightningTrainer(model, lr_info, trainer_options, temporal_test_dataset_parameters)

num_GPUs = torch.cuda.device_count()

# Define trainer
trainer = L.Trainer(accelerator="auto", 
                    devices=num_GPUs if num_GPUs > 0 else 'auto',
                    strategy='ddp_find_unused_parameters_true' if num_GPUs > 1 else 'auto',
                    max_epochs=1,
                    gradient_clip_val=1, 
                    precision='bf16-mixed')

# Predict and check probabilistic stuff

In [ ]:
prediction_file = os.path.join(save_folder, f"predictions.npy")
volume_file = os.path.join(save_folder, f"volumes.npy")

if not os.path.exists(prediction_file):
    prob_test_dataset = create_prob_test_dataset(base_datas, all_hydrographs, for_execution=True, temporal_res=temporal_res, scalers=scalers,
                            **temporal_test_dataset_parameters, **selected_node_features, **selected_edge_features)

    prob_test_dataloader = DataLoader(prob_test_dataset, batch_size=3, shuffle=False)

    start_time = time.time()
    predicted_rollout = trainer.predict(plmodel, dataloaders=prob_test_dataloader)
    prediction_times = time.time() - start_time
    predicted_rollout = [item for roll in predicted_rollout for item in roll]

    ensemble_selected_prediction = stack_rollout_different_BC(predicted_rollout, prob_test_dataset, scale=0)
    ensemble_predicted_volumes = torch.einsum('snt,n->st', ensemble_selected_prediction[:, :, 0].to(torch.float32), 
                                            torch.as_tensor(meshes[-1].face_area, device=ensemble_selected_prediction.device, dtype=torch.float32)).cpu().numpy()
    
    FAT_005 = torch.stack([WD_to_FAT(data[:, 0], temporal_res, 0.05, time_start) for data in ensemble_selected_prediction])
    FAT_03 = torch.stack([WD_to_FAT(data[:, 0], temporal_res, 0.3, time_start) for data in ensemble_selected_prediction])
    WD_max = torch.stack([data.max(2).values[:, 0] for data in ensemble_selected_prediction])
    V_max = torch.stack([data.max(2).values[:, 1] for data in ensemble_selected_prediction])
    WD_end = torch.stack([data[:, 0, -1] for data in ensemble_selected_prediction])

    ensemble_selected_prediction = torch.stack([FAT_005, FAT_03, WD_max, V_max, WD_end], dim=2).cpu().numpy()

    np.save(prediction_file, ensemble_selected_prediction)
    np.save(volume_file, ensemble_predicted_volumes)

else:
    ensemble_selected_prediction = np.load(prediction_file, mmap_mode='r')
    ensemble_predicted_volumes = np.load(volume_file, mmap_mode='r')

In [ ]:
prob_test_dataset = create_prob_test_dataset(base_datas, all_hydrographs, for_execution=True, temporal_res=temporal_res, scalers=scalers,
                         **temporal_test_dataset_parameters, **selected_node_features, **selected_edge_features)

### Summary all breaches

In [ ]:
base_DEM = base_datas[0].DEM if hasattr(base_datas[0], 'DEM') else None
base_mesh = meshes[-1]
gdf_mesh = meshes[0]

ARME_threshold = trainer_options.get("ARME_threshold", 0.4)

In [ ]:
fig = plot_breach_distribution_and_quantiles(ensemble_predicted_volumes, prob_test_dataset, num_breach_groups=8, 
                                                q_ranges=[0, 25, 50, 75, 100], max_ARME=2, show_percentage_runs=True, time_start=time_start)
# # plt.savefig(os.path.join(save_folder, 'breach_distribution_quantiles.png'), dpi=300, bbox_inches='tight')

In [ ]:
full_prob_analyser = ProbabilisticSpatialAnalysis(ensemble_selected_prediction, ensemble_predicted_volumes, prob_test_dataset,
                                                  scalers, base_DEM, base_mesh, **temporal_test_dataset_parameters)
plausible_runs = full_prob_analyser.get_plausible_runs(ARME_threshold=10)

In [ ]:
ax = plot_percentage_plausible_volumes_vs_ARME(full_prob_analyser.ARME, prob_test_dataset, ARME_thresholds=[1.0, 0.8, 0.6, 0.4, 0.2])

In [ ]:
fig, axs = plot_ARME_thresholds_analysis(full_prob_analyser, ensemble_selected_prediction, ensemble_predicted_volumes, prob_test_dataset, scalers, 
                                    temporal_test_dataset_parameters, ARME_thresholds=[0.4, 0.6, 1, 2])
# plt.savefig(os.path.join(save_folder, 'ARME_thresholds_analysis.png'), dpi=300, bbox_inches='tight')

In [ ]:
mask_ARME = full_prob_analyser.ARME < ARME_threshold

selected_prob_dataset = [prob_test_dataset[i] for i in np.where(mask_ARME)[0]]
prob_analyser = ProbabilisticSpatialAnalysis(ensemble_selected_prediction[mask_ARME], ensemble_predicted_volumes[mask_ARME],
                                             selected_prob_dataset, scalers, base_DEM, base_mesh, **temporal_test_dataset_parameters)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 6), gridspec_kw={'width_ratios': [1, 2]})

full_prob_analyser.plot_runs_per_threshold(with_hist=True, ARME_thresholds=np.linspace(0, max(ARME_threshold, 1.5), 50), ax=axs[0],
                                            bins=40, alpha=0.4, color='b', edgecolor='k', linewidth=0.5)
# axs[0].axvline(x=0.2, color='g', linestyle='--', linewidth=1.5)
axs[0].axvline(x=ARME_threshold, color='k', linestyle='--', linewidth=1.5)
# axs[0].axvline(x=1, color='r', linestyle='--', linewidth=1.5)

plausible_runs = full_prob_analyser.get_plausible_runs(ARME_threshold=ARME_threshold)
plausible_runs_per_location = np.bincount(plausible_runs // num_scenarios_per_location, minlength=num_breaches)
plot_valid_runs_breach_distribution(base_datas, plausible_runs_per_location/num_scenarios_per_location*100, cmap='YlGn', edgecolor='k', linewidth=0.3, ax=axs[1])
axs[1].set_aspect('auto')

panel_labels = [f'({chr(97+i)})' for i in range(len(axs))]
for i, label in enumerate(panel_labels):
    axs[i].text(0.02, 0.98, label, transform=axs[i].transAxes,
                fontsize=20, va='top', ha='left')
    
plt.tight_layout()

In [ ]:
fig, axs = plot_percentiles(ensemble_selected_prediction, quantiles=[0.25, 0.5, 0.75], variable='V_max', 
                                    temporal_res=temporal_res, **prob_analyser.default_plot_kwargs)

In [ ]:
ax = prob_analyser.plot_volume_distribution(log_scale=False, 
                                            # time_step=-1, 
                                            );

In [ ]:
mask_volume = full_prob_analyser.ARME >= ARME_threshold

anti_selected_prob_dataset = [prob_test_dataset[i] for i in np.where(mask_volume)[0]]
anti_prob_analyser = ProbabilisticSpatialAnalysis(ensemble_selected_prediction[mask_volume], ensemble_predicted_volumes[mask_volume],
                                             anti_selected_prob_dataset, scalers, base_DEM, base_mesh, **temporal_test_dataset_parameters)

In [ ]:
x, y = 175000, 428000

ax = add_inset_distribution_to_ax(prob_analyser.ensemble_selected_prediction[:,:,2], x, y, meshes, ax=None, bins=20)

### Explore single simulation

In [ ]:
id_dataset = 1

base_data = base_datas[id_dataset//num_breaches]

prob_test_dataset[id_dataset].edge_attr = get_edge_features(base_data, scalers=scalers, **selected_edge_features)
base_data.x = get_node_features(base_data, **selected_node_features, scalers=scalers)
    
WD = replicate_initial_condition(base_data.WD.unsqueeze(-1), previous_t)
V = replicate_initial_condition(base_data.V.unsqueeze(-1) ,previous_t)

prob_test_dataset[id_dataset].x = torch.cat([base_data.x, get_previous_steps(aggregate_WD_V, time_start, previous_t, WD, V)], dim=-1)
prob_test_dataset[id_dataset].y = torch.zeros(1, NUM_WATER_VARS, time_stop, device=device)

prob_test_dataset[id_dataset].DEM = base_data.DEM
prob_test_dataset[id_dataset].mesh = base_data.mesh

del prob_test_dataset[id_dataset].init_WD

rollout_plotter = SinglePlotRollout(plmodel, trainer, prob_test_dataset[id_dataset], 
                                    scalers=scalers, type_loss='RMSE', **temporal_test_dataset_parameters)

rollout_plotter.plot_BC();

In [ ]:
rollout_plotter._plot_volumes()
print("Volume R2:", rollout_plotter.get_volumes_r2())
print("Volume ARME:", rollout_plotter.get_volumes_ARME())

In [ ]:
rollout_plotter._mesh_scale_plot(scale=0)

rollout_plotter.predicted_WD.kwargs['vmax'] = 2

plot_times = [1, 2, 3, 4]
plot_times = [1, 5, 10, 30]
plot_times = [5, 10, 20, 30, 45]

rollout_plotter.show_h_rollout(plot_times=plot_times, scale=None, zoom_extent=None)
# rollout_plotter.show_v_rollout(plot_times=plot_times, scale=1, logscale=True, zoom_extent=None)
# rollout_plotter._plot_DEM()

In [ ]:
rollout_plotter._mesh_scale_plot(scale=0)

# rollout_plotter.pred_FATPlot.kwargs['vmax'] = 48

rollout_plotter.show_FAT(water_threshold=0.05, scale=0,
                        #  zoom_extent=rollout_plotter.around_breach
                         )